In [94]:
#https://docs.mistral.ai/capabilities/embeddings/rag_quickstart
from mistralai import Mistral
import requests
import numpy as np
import faiss  #pip intall faiss-cpu
import os
import json
from getpass import getpass

api_key= getpass("Type your API Key")
client = Mistral(api_key=api_key)

# Creating Embeddings and Storing them in a vector database

In this file, we first create embeddings from the `structured_chunks.json` file we obtained from `DataProcessing.ipynb` and then store them in a vector database to later use for similarity comparison.

First we load the chunks into python

In [95]:
# Pad naar je JSON-bestand
json_path = "structured_chunks.json"

# JSON-bestand inlezen
with open(json_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)
#data is nu een lijst van dictionaries. Iedere dictionary bevat tenminste de keys "text" en "filename" en mogelijk meer informatie.

In [96]:
print("Aantal chunks:", len(chunks))
print(chunks[0].keys())

Aantal chunks: 12060
dict_keys(['filename', 'hoofdstuk', 'text'])


Now we write a function that allows us to get embeddings from text and embed our chunks.

In [97]:
def get_text_embedding(input):
    embeddings_batch_response = client.embeddings.create(
          model="mistral-embed",
          inputs=input
      )
    return embeddings_batch_response.data[0].embedding


Unfortunately we have a rate limit of 20000000 tokens per minute, so let's introduce some delay between each request.
We start with putting a delay of 1 second between each request.

In [ ]:
import time
from tqdm import tqdm

chunks_length = len(chunks)
n_embeddings = chunks_length
embed_dim = 1024
save_path = "text_embeddings.npy"

# Laden van bestaande embeddings
if os.path.exists(save_path):
    text_embeddings = np.load(save_path)
    start_idx = np.count_nonzero(text_embeddings[:, 0])
    print(f"Herstart vanaf index {start_idx}/{chunks_length}")
else:
    text_embeddings = np.zeros((chunks_length, embed_dim))
    start_idx = 0

for i in tqdm(range(start_idx, chunks_length)):   #for i in tqdm(range(chunks_length)) for full embedding space
    time.sleep(1) #Wait 0.2 seconds between each request
    try:
        text_embeddings[i, :] = get_text_embedding(chunks[i].get("text"))
        # kleine vaste pauze om server te ontlasten
        time.sleep(1)

        if i % 100 == 0:
            np.save(save_path, text_embeddings)
    except Exception as e:
        print(f"Fout bij index {i}: {e}")
        np.save(save_path, text_embeddings)
        break

np.save(save_path, text_embeddings)
print("Klaar! Embeddings opgeslagen in", save_path)

  4%|▍         | 526/12060 [10:40<3:53:58,  1.22s/it]


KeyboardInterrupt: 

In [ ]:
import time
import numpy as np
from tqdm import tqdm
import os

chunks_length = len(chunks)
embed_dim = 1024
save_path = "text_embeddings.npy"

# Laden van bestaande embeddings
if os.path.exists(save_path):
    text_embeddings = np.load(save_path)
    start_idx = np.count_nonzero(text_embeddings[:, 0])
    print(f"Herstart vanaf index {start_idx}/{chunks_length}")
else:
    text_embeddings = np.zeros((chunks_length, embed_dim))
    start_idx = 0

def safe_get_text_embedding(text, max_retries=7):
    """Probeert embedding met exponentiële backoff."""
    delay = 1  # start met 1 sec
    for attempt in range(max_retries):
        try:
            return get_text_embedding(text)
        except Exception as e:
            msg = str(e).lower()
            if "capacity" in msg or "rate" in msg or "timeout" in msg:
                print(f"Rate limit / capacity issue (poging {attempt+1}), wacht {delay}s...")
                time.sleep(delay)
                delay *= 2  # verdubbel wachttijd
            else:
                raise  # echte fout
    raise RuntimeError(f"Embedding mislukt na {max_retries} pogingen.")

# Loop
for i in tqdm(range(start_idx, chunks_length)):
    try:
        text = chunks[i].get("text")
        text_embeddings[i, :] = safe_get_text_embedding(text)

        # kleine vaste pauze om server te ontlasten
        time.sleep(0.5)

        if i % 100 == 0:
            np.save(save_path, text_embeddings)

    except Exception as e:
        print(f"Fout bij index {i}: {e}")
        np.save(save_path, text_embeddings)
        break

np.save(save_path, text_embeddings)
print("Klaar! Embeddings opgeslagen in", save_path)


Herstart vanaf index 151/12060


  0%|          | 0/11909 [00:00<?, ?it/s]

Rate limit / capacity issue (poging 1), wacht 1s...
Rate limit / capacity issue (poging 2), wacht 2s...
Rate limit / capacity issue (poging 3), wacht 4s...


  0%|          | 6/11909 [00:41<15:18:25,  4.63s/it]

Rate limit / capacity issue (poging 1), wacht 1s...


...

Rate limit / capacity issue (poging 1), wacht 1s...


  1%|▏         | 155/11909 [17:56<27:29:07,  8.42s/it]

Rate limit / capacity issue (poging 1), wacht 1s...
Rate limit / capacity issue (poging 2), wacht 2s...
Rate limit / capacity issue (poging 3), wacht 4s...
Rate limit / capacity issue (poging 4), wacht 8s...
Rate limit / capacity issue (poging 5), wacht 16s...
Fout bij index 306: Embedding mislukt na 5 pogingen.


  1%|▏         | 155/11909 [18:53<23:52:45,  7.31s/it]


Klaar! Embeddings opgeslagen in text_embeddings.npy


Great, we have our text embeddings! Now let's store them in a vector database

# Load into Vector Database

Now that we have the embeddings, we want to load them into a vector database.

In [ ]:
d = text_embeddings.shape[1]
BeanRAG_VectorDB = faiss.IndexFlatL2(d)
BeanRAG_VectorDB.add(text_embeddings)

And then we store the vector database to be used in a different script.

In [53]:
print(BeanRAG_VectorDB.ntotal)
faiss.write_index(BeanRAG_VectorDB, "BeanRAG_VectorDB.faiss")
#To read the vector database, use faiss.read_index("BeanRAG_VectorDB.faiss")

50
